# 📊 Projeto Final: Análise Exploratória de Risco de Crédito
---
**Objetivo:** Investigar os determinantes da inadimplência e identificar perfis de risco (renda anual, faixa etária), visando otimizar a tomada de decisão na concessão de crédito.

**Autores:** Eduardo Roquette / Fernando Araújo

**Data:** Setembro/2026

In [ ]:
### DEFINIÇÃO DAS PERGUNTAS DE NEGÓCIOS:

# 1) Clientes com maior renda anual apresentam menores taxas de inadimplência?
# 2) O fator idade guarda correlação com índice de inadimplência?

In [ ]:
# IMPORT DAS BIBLIOTECAS / CONFIGURAÇÃOES PRÉVIAS / LEITURA DA BASE 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 130)

df_clientes = pd.read_csv('../dados/application_record.csv')
df_credito = pd.read_csv('../dados/credit_record.csv')

print("SHAPE | df_clientes:", df_clientes.shape)
print("SHAPE | df_credito:", df_credito.shape)

In [ ]:
# EXIBIÇÃO DAS INFORMAÇÕES GERAIS 

display(df_clientes.info(memory_usage='deep'))
display(df_credito.info(memory_usage='deep'))

In [ ]:
# RENOMEAÇÃO DAS COLUNAS EM DF_CLIENTES E DF_CREDITO

novos_nomes = {
    'ID': 'id_cliente',
    'CODE_GENDER': 'genero',
    'FLAG_OWN_CAR': 'possui_carro',
    'FLAG_OWN_REALTY': 'possui_imovel',
    'CNT_CHILDREN': 'qtd_filhos',
    'AMT_INCOME_TOTAL': 'renda_anual',
    'NAME_INCOME_TYPE': 'tipo_renda',
    'NAME_EDUCATION_TYPE': 'escolaridade',
    'NAME_FAMILY_STATUS': 'estado_civil',
    'NAME_HOUSING_TYPE': 'tipo_moradia',
    'DAYS_BIRTH': 'dias_nascimento',
    'DAYS_EMPLOYED': 'dias_empregado',
    'FLAG_MOBIL': 'possui_celular',
    'FLAG_WORK_PHONE': 'possui_tel_trabalho',
    'FLAG_PHONE': 'possui_tel_fixo',
    'FLAG_EMAIL': 'possui_email',
    'OCCUPATION_TYPE': 'ocupacao',
    'CNT_FAM_MEMBERS': 'qtd_membros_familia'
}

df_clientes = df_clientes.rename(columns=novos_nomes)

df_credito = df_credito.rename(columns={
    'ID': 'id_cliente',
    'MONTHS_BALANCE': 'mes_referencia',
    'STATUS': 'status_pagamento'
})

In [ ]:
# EXIBIÇÃO DAS INFORMAÇÕES GERAIS COM COLUNAS RENOMEADAS

display(df_clientes.info(memory_usage='deep'))
display(df_credito.info(memory_usage='deep'))

### === Investigação de Valores Nulos e Duplicados ===

In [ ]:
# INSPEÇÃO DE NAN POR COLUNAS

nulos_clientes = pd.DataFrame({
    'nulos_clientes': df_clientes.isna().sum(),
    'pct': (df_clientes.isna().mean() * 100).round(2)
})
nulos_clientes = nulos_clientes[nulos_clientes['nulos_clientes'] > 0].sort_values('nulos_clientes', ascending=False)
display(nulos_clientes)

nulos_credito = pd.DataFrame({
    'nulos_credito': df_credito.isna().sum(),
    'pct': (df_credito.isna().mean() * 100).round(2)
})
nulos_credito = nulos_credito[nulos_credito['nulos_credito'] > 0].sort_values('nulos_credito', ascending=False)
display(nulos_credito)

In [ ]:
# INSPEÇÃO DE NAN POR LINHAS

print("linhas com algum NaN --- base clientes --- base crédito:",  df_clientes.isna().any(axis=1).sum())
print("linhas completas --- base clientes:",  df_clientes.notna().all(axis=1).sum())

print("linhas com algum NaN --- base crédito:",  df_credito.isna().any(axis=1).sum())
print("linhas completas --- base crédito:",  df_credito.notna().all(axis=1).sum())

In [ ]:
# INSPEÇÃO DE REGISTROS DUPLICADOS

print("Valores duplicados | BASE CLIENTES:", df_clientes.duplicated().sum())
print("Valores duplicados | BASE CRÉDITO:", df_credito.duplicated().sum())

### === Reclassificação dos Registros das Colunas: OCUPAÇÃO, RENDA ANUAL, ESTADO CIVIL, MORADIA E ESCOLARIDADE ===

In [ ]:
df_clientes['ocupacao'].unique()

In [ ]:
# RENOMEAÇÃO DAS OCORRÊNCIAS EM OCUPAÇÃO

mapa_ocupacao = {
    'Security staff': 'Equipe de Segurança',
    'Sales staff': 'Vendas',
    'Accountants': 'Contadores',
    'Laborers': 'Operários',
    'Managers': 'Gerentes',
    'Drivers': 'Motoristas',
    'Core staff': 'Equipe Principal',
    'High skill tech staff': 'Equipe Técnica Especializada',
    'Cleaning staff': 'Equipe de Limpeza',
    'Private service staff': 'Serviços Privados',
    'Cooking staff': 'Cozinha',
    'Low-skill Laborers': 'Trabalhadores Operacionais',
    'Medicine staff': 'Equipe de Saúde',
    'Secretaries': 'Secretários(as)',
    'Waiters/barmen staff': 'Garçons/Bartenders',
    'HR staff': 'Recursos Humanos',
    'Realty agents': 'Corretores de Imóveis',
    'IT staff': 'Equipe de TI'
}

df_clientes['ocupacao'] = df_clientes['ocupacao'].replace(mapa_ocupacao)

In [ ]:
df_clientes['tipo_renda'].unique()

In [ ]:
# RENOMEAÇÃO DAS OCORRÊNCIAS EM TIPO_RENDA

mapa_tipo_renda = {
    'Working': 'Empregado',
    'Commercial associate': 'Autônomo / Comercial',
    'Pensioner': 'Aposentado / Pensionista',
    'State servant': 'Servidor Público',
    'Student': 'Estudante'
}

df_clientes['tipo_renda'] = df_clientes['tipo_renda'].replace(mapa_tipo_renda)

In [ ]:
df_clientes['estado_civil'].unique()

In [ ]:
# RENOMEAÇÃO DAS OCORRÊNCIAS EM ESTADO_CIVIL

mapa_estado_civil = {
    'Married': 'Casado(a)',
    'Single / not married': 'Solteiro(a)',
    'Civil marriage': 'União Estável',
    'Separated': 'Separado(a)',
    'Widow': 'Viúvo(a)'
}

df_clientes['estado_civil'] = df_clientes['estado_civil'].replace(mapa_estado_civil)
print(df_clientes['estado_civil'].unique())

In [ ]:
df_clientes['tipo_moradia'].unique()

In [ ]:
# RENOMEAÇÃO DAS OCORRÊNCIAS EM TIPO_MORADIA

mapa_tipo_moradia = {
    'House / apartment': 'Casa / Apartamento Próprio',
    'With parents': 'Com os Pais',
    'Municipal apartment': 'Habitação Pública',
    'Rented apartment': 'Aluguel',
    'Office apartment': 'Alojamento da Empresa',
    'Co-op apartment': 'Cooperativa Habitacional'
}

df_clientes['tipo_moradia'] = df_clientes['tipo_moradia'].replace(mapa_tipo_moradia)
print(df_clientes['tipo_moradia'].unique())

In [ ]:
df_clientes['escolaridade'].unique()

In [ ]:
# RENOMEAÇÃO DAS OCORRÊNCIAS EM ESCOLARIDADE

mapa_escolaridade = {
    'Secondary / secondary special': 'Ensino Médio / Técnico',
    'Higher education': 'Ensino Superior',
    'Incomplete higher': 'Superior Incompleto',
    'Lower secondary': 'Ensino Fundamental',
    'Academic degree': 'Pós-Graduação',
}

# Supondo que a coluna se chame 'escolaridade' (ou 'education_type')
df_clientes['escolaridade'] = df_clientes['escolaridade'].replace(
    mapa_escolaridade
)

# Validação dos valores únicos após a alteração
print(df_clientes['escolaridade'].unique())

#### === Processo de limpeza e modelagem (df_clientes) ===

In [ ]:
#FAZENDO DESCRIBE DO df_clientes PARA VERIFICAR AS ESTATÍSTICAS DESCRITIVAS

df_clientes.describe().round(2)


In [ ]:
# CRIAÇÃO DE COLUNAS IDADE E ANOS_EMPREGO COM UTILIZAÇÃO DO MÉTODO CLIP PARA TRATAMENTO DA FLAG APOSENTADOS E DESEMPREGADOS

df_clientes["idade"] = (-df_clientes["dias_nascimento"] / 365.25).astype(int)
df_clientes["anos_emprego"] = (
    (-df_clientes["dias_empregado"] / 365.25).clip(lower=0).round(1)
)  # clip para tornar os valores resultantes negativos em 0 para alinhamento da análise.

In [ ]:
# FAZENDO DESCRIBE APENAS DAS COLUNAS RELEVANTES 

estatisticas_clientes = df_clientes[['qtd_filhos', 'renda_anual', 'idade', 'anos_emprego', 'qtd_membros_familia']].describe().round(2)
display(estatisticas_clientes)
print("Possível Outlier em Renda Anual: ", estatisticas_clientes.loc['max', 'renda_anual'])

In [ ]:
# APLICAÇÃO DO MÉTODO IQR PARA TRATAMENTO DE OUTLIERS EM RENDA_ANUAL

q1 = df_clientes['renda_anual'].quantile(0.25)
q3 = df_clientes['renda_anual'].quantile(0.75)

iqr = q3 - q1

limite_inf = q1 - 1.5 * iqr
limite_sup = q3 + 1.5 * iqr

df_clientes = df_clientes[(df_clientes['renda_anual'] >= limite_inf) & (df_clientes['renda_anual'] <= limite_sup)]

In [ ]:
# ESTATÍSTICA DESCRITICA DA RENDA_ANUAL SEM OUTLIERS

df_clientes['renda_anual'].describe().round(2)

In [ ]:
# CONTAGEM DE TIPO DE RENDA PARA OCUPAÇÃO NAN

display(df_clientes[df_clientes['ocupacao'].isna()]['tipo_renda'].value_counts())
print("Total de nulos calculado em ocupação:", df_clientes[df_clientes['ocupacao'].isna()]['tipo_renda'].value_counts().sum())

In [ ]:
# IMPUTANDO VALORES NAN EM OCUPAÇÃO UTILIZANDO CORRESPONDENTE EM TIPO_RENDA

condicoes = [
    (df_clientes["ocupacao"].isna())
    & (df_clientes["tipo_renda"] == "Pensionista"),
    (df_clientes["ocupacao"].isna()),
]

escolhas = ["Pensionista", "Nao especificado"]

df_clientes["ocupacao"] = np.select(
    condicoes, escolhas, default=df_clientes["ocupacao"]
)

In [ ]:
# VERIFICAÇÃO DE NULOS APÓS IMPUTAÇÃO EM CLIENTES

print("Valores nulos | BASE CLIENTES: ", df_clientes['ocupacao'].isna().sum())

### === Processo de limpeza e modelagem (df_credito) ===

In [ ]:
display(df_credito['status_pagamento'].unique())

In [ ]:
# ==============================================================================
# DICIONÁRIO DE STATUS DE PAGAMENTO / ATRASO (STATUS)
# ------------------------------------------------------------------------------
# C | Pago em dia / Empréstimo quitado no mês
# X | Nenhum empréstimo ativo no mês
# 0 | 1 a 29 dias de atraso
# 1 | 30 a 59 dias de atraso
# 2 | 60 a 89 dias de atraso
# 3 | 90 a 119 dias de atraso
# 4 | 120 a 149 dias de atraso
# 5 | 150+ dias de atraso ou dívida baixada/prejuízo
# ==============================================================================

In [ ]:
# CRIAÇÃO DE COLUNA INADIMPLENTE (ATRASO >30 DIAS ---STATUS 1 A 5--- EM QUALQUER MêS)
inadimplente = ['1', '2', '3', '4', '5']
df_credito['inadimplente'] = df_credito['status_pagamento'].isin(inadimplente).astype(int)

In [ ]:
# VALIDAÇÃO: 1 = JÁ TEVE ATRASO GRAVE || 0 = NUNCA ATRASOU >30 DIAS

print(df_credito['inadimplente'].nunique())
print(df_credito['inadimplente'].unique())

In [ ]:
# AGRUPAMENTO DE CLIENTES QUE TIVERAM PELO MENOS 1 ATRASO GRAVE E CLIENTES BONS PAGADORES 

df_target = df_credito.groupby("id_cliente")["inadimplente"].max().reset_index()
display(df_target)
print("SHAPE | DF_TARGET:", df_target.shape, "SHAPE | DF_CLIENTES LIMPA:", df_clientes.shape)

In [ ]:
df_target['id_cliente'].nunique()

### === Cruzamento da Tabela Clientes com a Tabela Target ===

In [ ]:
# MERGE DAS BASES TARGET E CLIENTES COM TIPO INNER

df_final = pd.merge(df_clientes, df_target, on='id_cliente', how='inner')
print("SHAPE | DF_FINAL:", df_final.shape)

In [ ]:
display(df_final.head())

### === PERGUNTA 1: Faixa de Renda vs Inadimplência ===

In [ ]:
# DEFINIÇÃO DA FAIXA DE RENDA EM 5 PARTES IGUAIS + CÁLCULO DA TAXA DE INADIMPLÊNCIA POR FAIXA DE RENDA

df_final['faixa_renda'] = pd.cut(
    df_final['renda_anual'], 
    bins=5, 
    labels=['Muito Baixa', 'Baixa', 'Média', 'Alta', 'Muito Alta']
)

df_renda = (
    df_final.groupby('faixa_renda', observed=False)['inadimplente']
    .agg(total_clientes='count', taxa_inadimplencia='mean')
    .reset_index()
)

df_renda['taxa_inadimplencia'] = (df_renda['taxa_inadimplencia'] * 100).round(2).astype(str) + '%'

print(df_renda.to_string(index=False))

print("\n\n--- CONCLUSÕES: ---\n")
print("### 1. O desdobramento em 5 faixas confirma que o valor absoluto da renda não dita o risco de forma linear.")
print("### 2. Clientes da faixa 'Muito Alta' não são imunes à inadimplência; eles tendem a possuir limites maiores e alavancar mais suas faturas.")

In [ ]:
# GRÁFICO DE BARRAS

df_renda_grafico = (
    df_final.groupby('faixa_renda', observed=False)['inadimplente']
    .mean()
    .reset_index()
)

df_renda_grafico['taxa_inadimplencia'] = df_renda_grafico['inadimplente'] * 100

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=df_renda_grafico,
    x='faixa_renda',
    y='taxa_inadimplencia',
    hue='faixa_renda',       
    palette='viridis',       
    legend=False
)

plt.title('Taxa de Inadimplência por Faixa de Renda', fontsize=14, pad=15)
plt.xlabel('Quintis de Renda', fontsize=12, labelpad=10)
plt.ylabel('Taxa de Inadimplência (%)', fontsize=12, labelpad=10)

for p in ax.patches:
    altura = p.get_height()
    ax.annotate(f'{altura:.2f}%',
                (p.get_x() + p.get_width() / 2., altura),
                ha='center', va='bottom',
                fontsize=11, color='black', 
                xytext=(0, 5), textcoords='offset points')

sns.despine()
plt.show()

### === PERGUNTA 2: Idade vs Inadimplência ===

In [ ]:
# CÁLCULO DA CORRELAÇÃO DE PEARSON ENTRE A IDADE E INADIMPLENTE

df_final[['idade', 'inadimplente']].corr()

In [ ]:
df_idade_inadimplencia = df_final.groupby('idade')['inadimplente'].mean().reset_index()
df_idade_inadimplencia['taxa_inadimplencia'] = df_idade_inadimplencia['inadimplente'] * 100

plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_idade_inadimplencia,
    x='idade',
    y='taxa_inadimplencia',
    scatter_kws={'alpha':0.6},
    line_kws={'color': 'red'}
)

plt.title('Correlação Linear entre Idade e Taxa de Inadimplência', fontsize=14, pad=15)
plt.xlabel('Idade', fontsize=12, labelpad=10)
plt.ylabel('Taxa de Inadimplência (%)', fontsize=12, labelpad=10)
sns.despine()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("\n\n--- CONCLUSÃO: ---\n")
print("### A análise visual demonstra uma correlação linear negativa entre as variáveis.\nO risco de crédito apresenta uma tendência de queda à medida que a idade do cliente avança,\no que indica que perfis mais jovens concentram as maiores taxas de inadimplência,\nenquanto o comportamento de pagamento se torna progressivamente mais seguro e estável nas faixas etárias mais altas.")